In [160]:
import torch
from sklearn.cluster import KMeans
# Setup

%load_ext autoreload
%autoreload 2

from transformers import AutoModel, AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

print(f"HuggingFace cache directory is {os.environ.get('HF_HOME')}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
HuggingFace cache directory is /media/martin/BigData/datasets/cache/huggingface/


In [288]:
from pipelines.multi_service.gap_statistic import (
    calculate_gap_statistic,
    get_optimal_k_programmatically,
)
from numpy.random import uniform
from random import sample
from sklearn.neighbors import NearestNeighbors
import math
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import numpy as np

# Tokenize inputs

# MODEL = "microsoft/codebert-base"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)

input_texts = [
    "User createUser(User user)",
    "User updateUser(Long userId, User user)",
    "void deleteUser(Long userId)",
    "User getUserById(Long userId)",
    "List<User> listAllUsers()",
    "void activateUser(Long userId)",
    "void deactivateUser(Long userId)",
    "void changeUserPassword(Long userId, String newPassword)",
    "void assignUserRole(Long userId, String role)",
    "void removeUserRole(Long userId, String role)",
    # "Potion brewMagicPotion(Ingredient ingredient)",
    # "Elixir brewHealingElixir(Ingredient ingredient)",
    # "Draught brewStaminaDraught(Ingredient ingredient)",
    # "Infusion brewManaInfusion(Ingredient ingredient)",
    # "Potion brewFireResistancePotion(Ingredient ingredient)",
    # "Serum brewInvisibilitySerum(Ingredient ingredient)",
    # "Powder mixExplosivePowder(Ingredient ingredient)",
    # "WeatherReport fetchWeatherUpdate(Location location)",
    # "Gate allocateDepartureGate(Aircraft aircraft)",
    # "Gate releaseDepartureGate(Gate gate)",
    # "Runway reserveRunway(RunwayRequest request)",
    # "Runway releaseRunway(Runway runway)",
    # "LandingReport recordLandingData(LandingData data)",
    # "TakeoffReport recordTakeoffData(TakeoffData data)",
    # "FuelReport calculateFuelConsumption(FlightData data)",
    # "Checklist completeSafetyChecklist(Aircraft aircraft)",
    # "MaintenanceReport performMaintenance(Aircraft aircraft)",
    # "RefuelingTicket refuelAircraft(Aircraft aircraft)",
    # "BoardingPass boardPassenger(Passenger passenger)",
    # "PassengerManifest loadPassengerManifest(Manifest manifest)",
    # "CargoManifest loadCargoManifest(Cargo cargo)",
    # "Plane updateFlightStatus(FlightStatus status)",
    # "Route optimizeFlightRoute(Route route)",
]


# input_texts = [
#     "createUser",
#     "updateUser",
#     "deleteUser",
#     "getUserById",
#     "listAllUsers",
#     "activateUser",
#     "deactivateUser",
#     "changeUserPassword",
#     "assignUserRole",
#     "removeUserRole",
#     "brewMagicPotion",
#     "brewHealingElixir",
#     "brewStaminaDraught",
#     "brewManaInfusion",
#     "brewFireResistancePotion",
#     "brewInvisibilitySerum",
#     "mixExplosivePowder",
#     "fetchWeatherUpdate",
#     "allocateDepartureGate",
#     "releaseDepartureGate",
#     "reserveRunway",
#     "releaseRunway",
#     "recordLandingData",
#     "recordTakeoffData",
#     "calculateFuelConsumption",
#     "completeSafetyChecklist",
#     "performMaintenance",
#     "refuelAircraft",
#     "boardPassenger",
#     "loadPassengerManifest",
#     "loadCargoManifest",
#     "updateFlightStatus",
#     "optimizeFlightRoute",
# ]
#
# input_texts = [
#     "create order",
#     "update order",
#     "delete order",
#     "confirm order",
#     "cancel order",
#     "get order by id",
#     "get orders by customer id",
#     "list all orders",
#     "list open orders",
#     "list closed orders",
#     "calculate order total",
#     "add order item",
#     "remove order item",
#     "update order item quantity",
#     "apply discount to order",
#     "set order status",
#     "mark order as paid",
#     "mark order as shipped",
#     "mark order as delivered",
#     "mark order as returned"
# ]

input_texts = list(
    map(lambda text: "class EmployeeService { " + text + " {...} }", input_texts)
)

tokenized_inputs = tokenizer(
    input_texts, return_tensors="pt", padding=True, truncation=True, max_length=256
)
with torch.no_grad():
    output = model(**tokenized_inputs)


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output["last_hidden_state"]
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    # masked mean
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask


def hopkins(X):
    d = X.shape[1]
    # d = len(vars) # columns
    n = len(X)  # rows
    nbrs = NearestNeighbors(n_neighbors=1).fit(X)

    rand_X = sample(range(0, n, 1), n)

    ujd = []
    wjd = []
    for j in range(0, n):
        u_dist, _ = nbrs.kneighbors(
            uniform(np.amin(X, axis=0), np.amax(X, axis=0), d).reshape(1, -1),
            2,
            return_distance=True,
        )
        ujd.append(u_dist[0][1])
        w_dist, _ = nbrs.kneighbors(
            X[rand_X[j]].reshape(1, -1), 2, return_distance=True
        )
        wjd.append(w_dist[0][1])

    H = sum(ujd) / (sum(ujd) + sum(wjd))
    if math.isnan(H):
        print(ujd, wjd)
        H = 0

    return H


attention_mask = tokenized_inputs["attention_mask"]
# print(token_embeddings.shape)
# print(attention_mask)
# print(attention_mask.sum(axis=-1))
# print(attention_mask.sum(axis=-1).unsqueeze(-1))
# print(token_embeddings.sum(axis=1))
mean_pooled = mean_pooling(output, attention_mask)
# print(mean_pooled)

mean_pooled_numpy = mean_pooled.cpu().numpy()
# print(mean_pooled_numpy)


def entropy(counts):
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


pca = PCA(n_components=0.95, random_state=42)
pca_reduced = pca.fit_transform(mean_pooled_numpy)
print(pca_reduced.shape)

# print(f"Hopkins: {hopkins(pca_reduced)}")
clusters = [2, 3, 4, 5]

for k in clusters:
    kmeans = KMeans(n_clusters=k, random_state=42)
    result = kmeans.fit_predict(pca_reduced)
    print(result)

    # labels_counts = np.zeros(10, dtype=int)
    #
    # for label in result:
    #     labels_counts[label] += 1
    #
    # print(labels_counts)
    # print(entropy(labels_counts))

    silh_result = silhouette_score(pca_reduced, result)

    print(f"""K = {k}, silhouette score: {silh_result}""")

gap_df = calculate_gap_statistic(pca_reduced, n_refs=5, max_k=8)

# Find the first k where the criterion is met
# We look for the first positive value in 'diff' (or simply max gap if strict rule fails)
print(gap_df.head(8))
gap_result = get_optimal_k_programmatically(gap_df)
print(gap_result)

(10, 8)
[0 0 1 0 0 0 1 0 1 1]
K = 2, silhouette score: 0.0841129720211029
[2 0 1 0 2 1 1 1 0 1]
K = 3, silhouette score: 0.06881837546825409
[2 0 1 0 2 1 1 3 3 3]
K = 4, silhouette score: 0.13443180918693542
[2 0 1 0 2 1 1 4 3 3]
K = 5, silhouette score: 0.19300775229930878
   k       gap        sk   gap_k+1    sk_k+1      diff
0  1 -0.163510  0.155109 -0.309692  0.220377  0.366559
1  2 -0.309692  0.220377 -0.395551  0.214941  0.300800
2  3 -0.395551  0.214941 -0.382054  0.165529  0.152032
3  4 -0.382054  0.165529 -0.342466  0.189270  0.149682
4  5 -0.342466  0.189270 -0.355373  0.179106  0.192013
5  6 -0.355373  0.179106 -0.422385  0.190189  0.257202
6  7 -0.422385  0.190189 -0.515476  0.185039  0.278129
7  8 -0.515476  0.185039       NaN       NaN       NaN
{'optimal_k': 1, 'is_clusterable': False, 'reason': "1-Standard-Error Rule (First k satisfying Tibshirani's criterion)"}


In [180]:
# Tokenize inputs

MODEL = "microsoft/codebert-base"


# def tokenize_inputs(inputs: list[str]):
#     return map(lambda text_input: tokenizer(text_input, return_tensors="pt"), inputs)

# input_texts = [
#     "UserService[SEP]void getByID()",
#     "UserService[SEP]void getByName()",
#     "UserService[SEP]void ohHello()"
# ]

# input_texts = [
#     "User createUser(User user)",
#     "User updateUser(Long userId, User user)",
#     "void deleteUser(Long userId)",
#     "User getUserById(Long userId)",
#     "List<User> listAllUsers()",
#     "void activateUser(Long userId)",
#     "void deactivateUser(Long userId)",
#     "void changeUserPassword(Long userId, String newPassword)",
#     "void assignUserRole(Long userId, String role)",
#     "void removeUserRole(Long userId, String role)",
#
#     "LaunchSequence launchRocket(Rocket rocket)",
#     "Potion brewMagicPotion(Ingredient ingredient)"
# ]

# input_texts = [
#     "// Method name: createUser",
#     "// Method name: updateUser",
#     "// Method name: deleteUser",
#     "// Method name: getUserById",
#     "// Method name: listAllUsers",
#     "// Method name: activateUser",
#     "// Method name: deactivateUser",
#     "// Method name: changeUserPassword",
#     "// Method name: assignUserRole",
#     "// Method name: removeUserRole",
#     "// Method name: getLaunchedRocket",
#     "// Method name: getBrewedMagicPotion",
#     "UserService"
# ]

# input_texts = [
#     "// Class: EShopService\n// Method: createOrder",
#     "// Class: EShopService\n// Method: confirmOrder",
#     "// Class: EShopService\n// Method: deleteOrder",
#     "// Class: EShopService\n// Method: getOrders",
#     "// Class: EShopService\n// Method: getOrderById",
#     "// Class: EShopService\n// Method: createProduct",
#     "// Class: EShopService\n// Method: getProducts",
#     "// Class: EShopService\n// Method: getProductById",
#     "// Class: EShopService\n// Method: updateProduct",
#     "// Class: EShopService\n// Method: deleteProduct",
#     "// Class: EShopService\n// Method: addCustomer",
#     "// Class: EShopService\n// Method: getCustomers",
#     "// Class: EShopService\n// Method: getCustomerById",
#     "// Class: EShopService\n// Method: removeCustomer",
# ]

# input_texts = [
#     "// Class: OrderService\n// Method: createOrder",
#     "// Class: OrderService\n// Method: updateOrder",
#     "// Class: OrderService\n// Method: deleteOrder",
#     "// Class: OrderService\n// Method: confirmOrder",
#     "// Class: OrderService\n// Method: cancelOrder",
#     "// Class: OrderService\n// Method: getOrderById",
#     "// Class: OrderService\n// Method: getOrdersByCustomerId",
#     "// Class: OrderService\n// Method: listAllOrders",
#     "// Class: OrderService\n// Method: listOpenOrders",
#     "// Class: OrderService\n// Method: listClosedOrders",
#     "// Class: OrderService\n// Method: calculateOrderTotal",
#     "// Class: OrderService\n// Method: addOrderItem",
#     "// Class: OrderService\n// Method: removeOrderItem",
#     "// Class: OrderService\n// Method: updateOrderItemQuantity",
#     "// Class: OrderService\n// Method: applyDiscountToOrder",
#     "// Class: OrderService\n// Method: setOrderStatus",
#     "// Class: OrderService\n// Method: markOrderAsPaid",
#     "// Class: OrderService\n// Method: markOrderAsShipped",
#     "// Class: OrderService\n// Method: markOrderAsDelivered",
#     "// Class: OrderService\n// Method: markOrderAsReturned"
# ]

input_texts = [
    "create order",
    "update order",
    "delete order",
    "confirm order",
    "cancel order",
    "get order by id",
    "get orders by customer id",
    "list all orders",
    "list open orders",
    "list closed orders",
    "calculate order total",
    "add order item",
    "remove order item",
    "update order item quantity",
    "apply discount to order",
    "set order status",
    "mark order as paid",
    "mark order as shipped",
    "mark order as delivered",
    "mark order as returned",
]

# input_texts = list(map(lambda text: "class UserService { " + text + " {...} }", input_texts))


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[
        0
    ]  # First element of model_output contains all token embeddings
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )


# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# Tokenize sentences
encoded_input = tokenizer(
    input_texts, padding=True, truncation=True, return_tensors="pt"
)

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling
sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

# Normalize embeddings
# sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
mean_pooled_numpy = sentence_embeddings.cpu().numpy()
print("Sentence embeddings:")
print(sentence_embeddings)


# K = math.ceil(math.sqrt(len(input_texts)))
K = 2
kmeans = KMeans(n_clusters=K, random_state=42)
result = kmeans.fit_predict(mean_pooled_numpy)


# def entropy(counts):
#     total = counts.sum()
#     if total == 0:
#         return 0.0
#     p = counts / total
#     p = p[p > 0]
#     return np.sum(p * np.log2(p))


labels_counts = np.zeros(K, dtype=int)

for label in result:
    labels_counts[label] += 1

print(labels_counts)
print(entropy(labels_counts))
# cosine_result = pytorch_cos_sim(mean_pooled_numpy[10], mean_pooled_numpy[14])
# print(cosine_result)

Sentence embeddings:
tensor([[-0.3541, -0.2261,  0.0257,  ...,  0.0177,  0.3455, -0.0453],
        [-0.3325, -0.1847,  0.4260,  ..., -0.3638,  0.0852, -0.2764],
        [-0.3018,  0.2795,  0.2313,  ...,  0.1468,  0.1780, -0.3855],
        ...,
        [-0.4235,  0.0537,  0.3281,  ...,  0.0457,  0.1402, -0.0622],
        [-0.2836,  0.3735,  0.5110,  ..., -0.0335,  0.4223, -0.0301],
        [-0.1840,  0.3176,  0.6899,  ..., -0.2170,  0.0471,  0.0534]])
[ 9 11]
-0.9927744539878083
